In [118]:
import pandas as pd
import requests
import bs4 
import regex as re
import json

In [119]:
# Open the item summary url link and make it into beautiful soup
url = 'https://tesdatrainingcourses.com/list-of-tesda-short-courses-offered-in-training-centers.html#aff'
response = requests.get(url)
soup = bs4.BeautifulSoup(response.text)

# get the tables
tables = soup.findAll('table', class_='table sortable jquery-tablesorter')
courses = []

for table in tables:
    # get the courses
    sector = table.find_previous('p').get_text(" ", strip=True)
    simplified_sector = re.match(r"List of TESDA Courses(?: for)? (.+)", sector).group(1)

    for row in table.select('tbody tr'):
        cells = row.find_all('td')
        #print(link)

        # get the name and cells
        link = cells[0].find('a')
        if link:
            name = link.get_text("", strip=True)
            href = link.get("href")
        else:
            name = cells[0].get_text(" ", strip=True)
            href = None

        hours = cells[1].get_text(" ", strip=True)

        courses.append({
            'Sector': simplified_sector,
            'Course': name,
            'Link': href,
            'Hours': hours,
        })

In [120]:
for course in courses:
    # read the Link if possible. If none, skip
    url = course['Link']
    if not url:
        continue

    response = requests.get(url)
    soup = bs4.BeautifulSoup(response.text, 'html.parser')

    # get the table that contains the codes
    tables = soup.find_all(
        'table',
        class_='table sortable jquery-tablesorter'
    )

    # if there is no table, we will be having a problem. Print it out and skip
    if not tables:
        print(course)
        continue

    # get the first table that contains the codes
    table = tables[0]

    # set up the lists that will contain the codes
    course['Core'] = []
    course['Common'] = []
    course['Basic'] = []
    course['Elective'] = []

    # for loops dont have scope, so we will use that to our advatange
    competency_type = None
    
    for row in table.select('tr'):

        row_text = row.get_text(" ", strip=True)

        # Check whether this row defines a competency category
        match = re.search(
            r'\b(CORE|COMMON|BASIC|ELECTIVE)(\s+)?COMPETENCIES',
            row_text,
            re.IGNORECASE
        )

        # this would be the competency type and they follow over the next passes
        if match:
            competency_type = match.group(1).capitalize()
            continue
        else:
            # get the code and its description
            matches = re.search(r'^(\S+)\s+(.+)$', row_text)
            code = matches.group(1)
            description = matches.group(2)

            # add it to the course
            course[competency_type].append({code : description})

{'Sector': 'Information and Communication Technology', 'Course': '2D Animation NC III', 'Link': 'https://tesdatrainingcourses.com/tesda-animation-courses.html', 'Hours': '968'}


In [128]:
# now convert into a pandas dataframe
courses_df = pd.DataFrame(courses)

We will extract two things. 
1. Per NC, get its unit of competency codes
2. Per unit of competency code, get its description.

In [129]:
# flatten the codes
codes_columns = ['Core', 'Common', 'Basic', 'Elective']

def flatten_codes(row, codes_columns=codes_columns):
    """get the codes"""
    codes = []
    for code_col in codes_columns:
        if isinstance(row[code_col], list):
            for code_dict in row[code_col]:
                codes.extend(code_dict)
    return codes

courses_df['Codes'] = courses_df.apply(flatten_codes, axis=1)

In [130]:
courses_df.to_csv('../data/tesda/tesda_competencies.csv', index=False)

In [ ]:
# save as the json the mapping of courses to codes
nc_codes_mapping = dict(zip(courses_df.Course, courses_df.Codes))
filename = '../data/tesda/nc_course_codes_mapping.json'
with open(filename, 'w') as file:
    json.dump(nc_codes_mapping, file)

In [134]:
# now we get the dictionary per row
def extract_code_desc_mapping(row, codes_columns=codes_columns):
    """get the codes"""
    code_desc_mapping = {}
    for code_col in codes_columns:
        if isinstance(row[code_col], list):
            for code_dict in row[code_col]:
                for code, desc in code_dict.items():
                    code_desc_mapping[code] = desc
    return code_desc_mapping

courses_df['Code Dictionary'] = courses_df.apply(extract_code_desc_mapping, axis=1)

In [135]:
overall_code_dict = {}
for code_dict in courses_df['Code Dictionary']:
    overall_code_dict.update(code_dict)

In [136]:
filename = '../data/tesda/tr_code_desc_mapping.json'
with open(filename, 'w') as file:
    json.dump(overall_code_dict, file)